In [ ]:
!pip install gradio pandas numpy matplotlib

In [ ]:
!pip install gradio

In [78]:
import gradio as gr
import pandas as pd
import numpy as np
from io import BytesIO
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import datetime
import time

# ---------------- HELPERS ----------------
def normalize(series):
    # Ensure series is numeric and handle potential division by zero if max-min is 0
    if series.empty or series.max() == series.min():
        return pd.Series(0, index=series.index) # Return zeros or handle as appropriate
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def get_required_columns(business_type):
    base_columns = [
        'Monthly_Sales_INR', 'Monthly_Operating_Cost_INR', 'Outstanding_Loan_INR',
        'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent',
        'Monthly_Demand_Units', 'SKU_Name'
    ]
    if business_type == "FMCG":
        base_columns.extend(['Distribution_Reach', 'Promotion_Spend_INR'])
    elif business_type == "Supermarket":
        base_columns.extend(['Digital_Ad_Spend_INR'])
    elif business_type == "Clothing":
        base_columns.extend(['Season'])
    elif business_type == "Electronics":
        base_columns.extend(['Product_Age_Months', 'Customer_Segment', 'Season'])
    return base_columns

# ---------------- AI INSIGHTS ----------------
def generate_insights(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df):
    final_city = other_city_name if city == "Other" else city

    try:
        # Ensure columns are numeric, coercing errors to NaN and then filling
        for col in ['Monthly_Operating_Cost_INR', 'Monthly_Sales_INR', 'Outstanding_Loan_INR', 'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent', 'Monthly_Demand_Units']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0) # Fill NaN with 0 for calculations

        # Handle potential ZeroDivisionError by checking if denominator is zero
        if (df['Monthly_Sales_INR'] == 0).any():
             # Temporarily replace 0 with a small number to avoid division by zero, then calculate stress
            temp_monthly_sales = df['Monthly_Sales_INR'].replace(0, 1e-9)
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / temp_monthly_sales)
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (temp_monthly_sales*12))
        else:
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / df["Monthly_Sales_INR"])
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (df["Monthly_Sales_INR"]*12))

        df["Financial_Risk_Score"] = (0.5*df["Cashflow_Stress"] + 0.5*df["Loan_Stress"]).clip(0,1)
        df["Vendor_Score"] = (0.5*df["Vendor_Delivery_Reliability"] +
                              0.3*normalize(df["Inventory_Turnover"]) +
                              0.2*normalize(df["Avg_Margin_Percent"])) .clip(0,1)
        df["Growth_Potential_Score"] = (0.4*normalize(df["Monthly_Demand_Units"]) +
                                        0.35*normalize(df["Avg_Margin_Percent"]) +
                                        0.25*normalize(df.get("Digital_Ad_Spend_INR", pd.Series(0)))) .clip(0,1)
        df["MSME_Health_Score"] = ((1 - df["Financial_Risk_Score"])*0.4 +
                                   df["Vendor_Score"]*0.3 +
                                   df["Growth_Potential_Score"]*0.3)*100
        top_skus = df.sort_values(by="Monthly_Sales_INR", ascending=False).head(5)[["SKU_Name","Monthly_Sales_INR","Monthly_Demand_Units"]]

        insights = f"""
**DataNetra Insights for {user_name} – {company_name} ({business_type}) in {final_city}, {state}):**

- Average Financial Risk Score: {df["Financial_Risk_Score"].mean():.2f}
- Average Vendor Score: {df["Vendor_Score"].mean():.2f}
- Average Growth Potential Score: {df["Growth_Potential_Score"].mean():.2f}
- Overall MSME Health Score: {df["MSME_Health_Score"].mean():.2f}%

**Top 5 SKUs:**
{top_skus.to_string(index=False)}

**Recommendations:**
- Focus on top SKUs for sales growth.
- Monitor high financial risk MSMEs carefully.
- Ensure reliable vendors are prioritized.
- Optimize inventory and demand planning.
"""
        return insights, None # Return insights and no error
    except Exception as e:
        return None, f"Error generating AI Insights: {str(e)}. Please check your data for numerical issues or unexpected values."

# ---------------- PDF GENERATION ----------------
def generate_pdf(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df):
    final_city = other_city_name if city == "Other" else city
    pdf_buffer = BytesIO()

    try:
        # Ensure columns are numeric, coercing errors to NaN and then filling
        for col in ['Monthly_Operating_Cost_INR', 'Monthly_Sales_INR', 'Outstanding_Loan_INR', 'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent', 'Monthly_Demand_Units']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # Handle potential ZeroDivisionError by checking if denominator is zero
        if (df['Monthly_Sales_INR'] == 0).any():
            temp_monthly_sales = df['Monthly_Sales_INR'].replace(0, 1e-9)
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / temp_monthly_sales)
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (temp_monthly_sales*12))
        else:
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / df["Monthly_Sales_INR"])
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (df["Monthly_Sales_INR"]*12))

        df["Financial_Risk_Score"] = (0.5*df["Cashflow_Stress"] + 0.5*df["Loan_Stress"]).clip(0,1)
        df["Vendor_Score"] = (0.5*df["Vendor_Delivery_Reliability"] +
                              0.3*normalize(df["Inventory_Turnover"]) +
                              0.2*normalize(df["Avg_Margin_Percent"])) .clip(0,1)
        df["Growth_Potential_Score"] = (0.4*normalize(df["Monthly_Demand_Units"]) +
                                        0.35*normalize(df["Avg_Margin_Percent"]) +
                                        0.25*normalize(df.get("Digital_Ad_Spend_INR", pd.Series(0)))) .clip(0,1)
        df["MSME_Health_Score"] = ((1 - df["Financial_Risk_Score"])*0.4 +
                                   df["Vendor_Score"]*0.3 +
                                   df["Growth_Potential_Score"]*0.3)*100

        top_skus = df.sort_values(by="Monthly_Sales_INR", ascending=False).head(5)[["SKU_Name","Monthly_Sales_INR","Monthly_Demand_Units"]]

        # Forecast calculations
        years = [1,2,3]
        sales_growth = 1 + (df["Growth_Potential_Score"].mean() * 0.15)
        demand_growth = 1 + (df["Growth_Potential_Score"].mean() * 0.12)
        avg_margin = df["Avg_Margin_Percent"].mean() / 100

        sales_forecast = [(df["Monthly_Sales_INR"].sum()*12)*(sales_growth**i) for i in years]
        demand_forecast = [df["Monthly_Demand_Units"].sum()*(demand_growth**i) for i in years]
        profit_forecast = [sales_forecast[i]*avg_margin for i in range(len(years))]

        with PdfPages(pdf_buffer) as pdf:
            # Page 1 – Cover
            plt.figure(figsize=(8,6)); plt.axis("off")
            plt.text(0.5,0.6,"DataNetra – AI Decision Support Report", fontsize=18, ha='center')
            plt.text(0.5,0.5,f"{user_name} | {company_name} | {business_type}", fontsize=12, ha='center')
            plt.text(0.5,0.4,f"{final_city}, {state} | {datetime.datetime.today().strftime('%d-%b-%Y')}", fontsize=10, ha='center')
            pdf.savefig(); plt.close()

            # Page 2 – Table of Contents
            toc = ["1. Executive Summary","2. Financial Risk Analysis","3. Vendor Analysis","4. Growth & Forecast",
                   "5. Top SKUs","6. Actionable Insights & Conclusion"]
            plt.figure(figsize=(8,6)); plt.axis("off")
            y = 0.9
            plt.text(0.5,0.9,"Table of Contents", fontsize=16, ha='center')
            for i, item in enumerate(toc):
                y -= 0.1
                plt.text(0.1,y,f"{item} ...... Page {i+3}", fontsize=12)
            pdf.savefig(); plt.close()

            # Page 3 – Executive Summary
            plt.figure(figsize=(8,6)); plt.axis("off")
            summary_text = f"""
Executive Summary:
- Overall MSME Health Score: {df["MSME_Health_Score"].mean():.2f}%
- Financial Risk Avg: {df["Financial_Risk_Score"].mean():.2f}
- Vendor Score Avg: {df["Vendor_Score"].mean():.2f}
- Growth Potential Avg: {df["Growth_Potential_Score"].mean():.2f}
"""
            plt.text(0.05,0.5,summary_text, fontsize=12)
            pdf.savefig(); plt.close()

            # Page 4 – Financial Risk Analysis
            plt.figure(); plt.hist(df["Financial_Risk_Score"], bins=10, color="red")
            plt.title("Financial Risk Score Distribution"); plt.xlabel("Score"); plt.ylabel("Count"); plt.grid(True)
            pdf.savefig(); plt.close()

            plt.figure(); plt.plot(years, sales_forecast, marker="o", color="blue"); plt.title("3-Year Sales Forecast"); plt.xlabel("Year"); plt.ylabel("Sales INR"); plt.grid(True)
            pdf.savefig(); plt.close()

            plt.figure(); plt.plot(years, demand_forecast, marker="o", color="orange"); plt.title("3-Year Demand Forecast"); plt.xlabel("Year"); plt.ylabel("Demand Units"); plt.grid(True)
            pdf.savefig(); plt.close()

            plt.figure(); plt.plot(years, profit_forecast, marker="o", color="purple"); plt.title("3-Year Profit Forecast"); plt.xlabel("Year"); plt.ylabel("Profit INR"); plt.grid(True)
            pdf.savefig(); plt.close()

            # Page 7 – Top SKUs
            plt.figure(figsize=(8,2)); plt.axis("off")
            plt.text(0.01,0.5,"Top 5 SKUs:\n"+top_skus.to_string(index=False), fontsize=10)
            pdf.savefig(); plt.close()

            # Page 8 – Conclusion / Recommendations
            plt.figure(figsize=(8,6)); plt.axis("off")
            conclusion_text = """Conclusion & Actionable Insights:
- Focus on top SKUs for sales growth.
- Monitor high financial risk MSMEs carefully.
- Prioritize reliable vendors.
- Optimize inventory and demand planning.
"""
            plt.text(0.05,0.5,conclusion_text, fontsize=12)
            pdf.savefig(); plt.close()

        pdf_buffer.seek(0)
        pdf_path = f"/tmp/{business_type}_{user_name}_report.pdf".replace(" ","_")
        with open(pdf_path, "wb") as f:
            f.write(pdf_buffer.getbuffer())
        return pdf_path, None # Return path and no error
    except Exception as e:
        return None, f"Error generating PDF report: {str(e)}. Please check your data for numerical issues or unexpected values."


# ---------------- DASHBOARD DATA GENERATION ----------------
def generate_dashboard_data(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df):
    final_city = other_city_name if city == "Other" else city

    try:
        # Ensure columns are numeric, coercing errors to NaN and then filling
        for col in ['Monthly_Operating_Cost_INR', 'Monthly_Sales_INR', 'Outstanding_Loan_INR', 'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent', 'Monthly_Demand_Units']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # Handle potential ZeroDivisionError by checking if denominator is zero
        if (df['Monthly_Sales_INR'] == 0).any():
            temp_monthly_sales = df['Monthly_Sales_INR'].replace(0, 1e-9)
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / temp_monthly_sales)
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (temp_monthly_sales*12))
        else:
            df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / df["Monthly_Sales_INR"])
            df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (df["Monthly_Sales_INR"]*12))

        df["Financial_Risk_Score"] = (0.5*df["Cashflow_Stress"] + 0.5*df["Loan_Stress"]).clip(0,1)
        df["Vendor_Score"] = (0.5*df["Vendor_Delivery_Reliability"] +
                              0.3*normalize(df["Inventory_Turnover"]) +
                              0.2*normalize(df["Avg_Margin_Percent"])) .clip(0,1)
        df["Growth_Potential_Score"] = (0.4*normalize(df["Monthly_Demand_Units"]) +
                                        0.35*normalize(df["Avg_Margin_Percent"]) +
                                        0.25*normalize(df.get("Digital_Ad_Spend_INR", pd.Series(0)))) .clip(0,1)
        df["MSME_Health_Score"] = ((1 - df["Financial_Risk_Score"])*0.4 +
                                   df["Vendor_Score"]*0.3 +
                                   df["Growth_Potential_Score"]*0.3)*100

        # KPIs
        total_sales = df["Monthly_Sales_INR"].sum()
        total_profit = (df["Monthly_Sales_INR"] * (df["Avg_Margin_Percent"] / 100)).sum()
        msme_health_score = df["MSME_Health_Score"].mean()
        growth_potential = df["Growth_Potential_Score"].mean()

        # Forecast data (re-using calculations from PDF generation)
        years = [1,2,3]
        sales_growth_factor = 1 + (df["Growth_Potential_Score"].mean() * 0.15)
        demand_growth_factor = 1 + (df["Growth_Potential_Score"].mean() * 0.12)
        avg_margin = df["Avg_Margin_Percent"].mean() / 100

        sales_forecast = [df["Monthly_Sales_INR"].sum() * (sales_growth_factor**i) for i in years]
        profit_forecast = [sales_forecast[i] * avg_margin for i in range(len(years))]

        # Generate 4 charts based on business type
        fig1, fig2, fig3, fig4 = plt.figure(), plt.figure(), plt.figure(), plt.figure() # Initialize empty figures

        if business_type == "Supermarket":
            # Chart 1: Sales by SKU (Top N)
            fig1, ax1 = plt.subplots(figsize=(6, 4))
            df.groupby("SKU_Name")["Monthly_Sales_INR"].sum().nlargest(5).plot(kind="bar", ax=ax1, color="skyblue")
            ax1.set_title("Top 5 SKUs by Sales")
            ax1.set_xlabel("SKU Name")
            ax1.set_ylabel("Monthly Sales (INR)")
            plt.tight_layout()
            plt.close(fig1)

            # Chart 2: Inventory Turnover
            fig2, ax2 = plt.subplots(figsize=(6, 4))
            df.plot(kind='scatter', x='Inventory_Turnover', y='Monthly_Sales_INR', ax=ax2, color='orange') # Changed x-axis to Inventory_Turnover
            ax2.set_title("Sales vs. Inventory Turnover") # Changed title
            ax2.set_xlabel("Inventory Turnover")
            ax2.set_ylabel("Monthly Sales (INR)")
            plt.tight_layout()
            plt.close(fig2)

            # Chart 3: Average Margin Percent Distribution
            fig3, ax3 = plt.subplots(figsize=(6, 4))
            ax3.hist(df["Avg_Margin_Percent"], bins=10, color="lightgreen") # Changed to histogram for distribution
            ax3.set_title("Average Margin Percent Distribution") # Changed title
            ax3.set_xlabel("Average Margin (%)")
            ax3.set_ylabel("Count")
            plt.tight_layout()
            plt.close(fig3)

            # Chart 4: Sales vs. Digital Ad Spend (if data exists)
            fig4, ax4 = plt.subplots(figsize=(6, 4))
            if "Digital_Ad_Spend_INR" in df.columns and not df["Digital_Ad_Spend_INR"].isnull().all():
                ax4.scatter(df["Digital_Ad_Spend_INR"], df["Monthly_Sales_INR"], color="purple", alpha=0.7)
                ax4.set_title("Sales vs. Digital Ad Spend")
                ax4.set_xlabel("Digital Ad Spend (INR)")
                ax4.set_ylabel("Monthly Sales (INR)")
            else:
                ax4.text(0.5, 0.5, "Digital Ad Spend Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax4.transAxes, fontsize=12)
                ax4.set_xticks([])
                ax4.set_yticks([])
                ax4.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig4)

        elif business_type == "FMCG":
            # Chart 1: SKU Sales Volume (Top N)
            fig1, ax1 = plt.subplots(figsize=(6, 4))
            df.groupby("SKU_Name")["Monthly_Sales_INR"].sum().nlargest(5).plot(kind="bar", ax=ax1, color="skyblue")
            ax1.set_title("Top 5 SKUs by Sales Volume")
            ax1.set_xlabel("SKU Name")
            ax1.set_ylabel("Monthly Sales (INR)")
            plt.tight_layout()
            plt.close(fig1)

            # Chart 2: Growth Trends (Overall Sales Forecast)
            fig2, ax2 = plt.subplots(figsize=(6, 4))
            ax2.plot(years, sales_forecast, marker="o", color="blue")
            ax2.set_title("3-Year Sales Forecast")
            ax2.set_xlabel("Year")
            ax2.set_ylabel("Sales INR")
            ax2.grid(True)
            plt.tight_layout()
            plt.close(fig2)

            # Chart 3: Demand Prediction (Top 5 SKUs by Monthly Demand)
            fig3, ax3 = plt.subplots(figsize=(6, 4))
            df.groupby("SKU_Name")["Monthly_Demand_Units"].sum().nlargest(5).plot(kind="bar", ax=ax3, color="teal")
            ax3.set_title("Top 5 SKUs by Monthly Demand Units")
            ax3.set_xlabel("SKU Name")
            ax3.set_ylabel("Monthly Demand (Units)")
            plt.tight_layout()
            plt.close(fig3)

            # Chart 4: Distribution Efficiency (if 'Distribution_Reach' column exists)
            fig4, ax4 = plt.subplots(figsize=(6, 4))
            if "Distribution_Reach" in df.columns:
                df.groupby("Distribution_Reach")["Monthly_Sales_INR"].sum().plot(kind="pie", ax=ax4, autopct='%1.1f%%', startangle=90)
                ax4.set_title("Sales by Distribution Reach")
                ax4.set_ylabel('') # Hide y-label for pie chart
            else:
                ax4.text(0.5, 0.5, "Distribution Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax4.transAxes, fontsize=12)
                ax4.set_xticks([])
                ax4.set_yticks([])
                ax4.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig4)

        elif business_type == "Clothing":
            # Chart 1: Sales Analysis (Top 5 SKUs)
            fig1, ax1 = plt.subplots(figsize=(6, 4))
            df.groupby("SKU_Name")["Monthly_Sales_INR"].sum().nlargest(5).plot(kind="bar", ax=ax1, color="lightcoral")
            ax1.set_title("Top 5 SKUs by Sales")
            ax1.set_xlabel("SKU Name")
            ax1.set_ylabel("Monthly Sales (INR)")
            plt.tight_layout()
            plt.close(fig1)

            # Chart 2: Seasonality & Trend (if 'Season' column exists)
            fig2, ax2 = plt.subplots(figsize=(6, 4))
            if "Season" in df.columns:
                df.groupby("Season")["Monthly_Sales_INR"].sum().plot(kind="bar", ax=ax2, color="skyblue")
                ax2.set_title("Sales by Season")
                ax2.set_xlabel("Season")
                ax2.set_ylabel("Monthly Sales (INR)")
            else:
                ax2.text(0.5, 0.5, "Season Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes, fontsize=12)
                ax2.set_xticks([])
                ax2.set_yticks([])
                ax2.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig2)

            # Chart 3: Inventory Management (example: Inventory Turnover distribution)
            fig3, ax3 = plt.subplots(figsize=(6, 4))
            ax3.hist(df["Inventory_Turnover"], bins=10, color="orange")
            ax3.set_title("Inventory Turnover Distribution")
            ax3.set_xlabel("Inventory Turnover")
            ax3.set_ylabel("Count")
            plt.tight_layout()
            plt.close(fig3)

            # Chart 4: Pricing & Customer Behavior (Avg Margin Percent Distribution) - using existing logic
            fig4, ax4 = plt.subplots(figsize=(6, 4))
            ax4.hist(df["Avg_Margin_Percent"], bins=10, color="purple")
            ax4.set_title("Average Margin Percent Distribution")
            ax4.set_xlabel("Average Margin (%)")
            ax4.set_ylabel("Count")
            plt.tight_layout()
            plt.close(fig4)

        elif business_type == "Electronics":
            # Chart 1: Sales Metrics (Top 5 SKUs by sales)
            fig1, ax1 = plt.subplots(figsize=(6, 4))
            df.groupby("SKU_Name")["Monthly_Sales_INR"].sum().nlargest(5).plot(kind="bar", ax=ax1, color="darkblue")
            ax1.set_title("Top 5 SKUs by Sales")
            ax1.set_xlabel("SKU Name")
            ax1.set_ylabel("Monthly Sales (INR)")
            plt.tight_layout()
            plt.close(fig1)

            # Chart 2: Product Lifecycle (Sales vs. Product Age) - if column exists)
            fig2, ax2 = plt.subplots(figsize=(6, 4))
            if "Product_Age_Months" in df.columns:
                ax2.scatter(df["Product_Age_Months"], df["Monthly_Sales_INR"], color="red", alpha=0.7)
                ax2.set_title("Sales vs. Product Age (Months)")
                ax2.set_xlabel("Product Age (Months)")
                ax2.set_ylabel("Monthly Sales (INR)")
            else:
                ax2.text(0.5, 0.5, "Product Age Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax2.transAxes, fontsize=12)
                ax2.set_xticks([])
                ax2.set_yticks([])
                ax2.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig2)

            # Chart 3: Customer Preferences (if 'Customer_Segment' exists)
            fig3, ax3 = plt.subplots(figsize=(6, 4))
            if "Customer_Segment" in df.columns:
                df.groupby("Customer_Segment")["Monthly_Sales_INR"].sum().plot(kind="bar", ax=ax3, color="green")
                ax3.set_title("Sales by Customer Segment")
                ax3.set_xlabel("Customer Segment")
                ax3.set_ylabel("Monthly Sales (INR)")
            else:
                ax3.text(0.5, 0.5, "Customer Segment Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax4.transAxes, fontsize=12)
                ax3.set_xticks([])
                ax3.set_yticks([])
                ax3.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig3)

            # Chart 4: Seasonal & Event Trends (if 'Season' exists)
            fig4, ax4 = plt.subplots(figsize=(6, 4))
            if "Season" in df.columns:
                df.groupby("Season")["Monthly_Sales_INR"].sum().plot(kind="line", ax=ax4, marker="o", color="purple")
                ax4.set_title("Seasonal Sales Trends")
                ax4.set_xlabel("Season")
                ax4.set_ylabel("Monthly Sales (INR)")
            else:
                ax4.text(0.5, 0.5, "Season Data Not Available", horizontalalignment='center', verticalalignment='center', transform=ax4.transAxes, fontsize=12)
                ax4.set_xticks([])
                ax4.set_yticks([])
                ax4.set_frame_on(False)
            plt.tight_layout()
            plt.close(fig4)

        else: # Default case for other business types
            # Fallback to general sales and profit forecasts, and score distributions
            # Chart 1: Sales forecast
            fig1 = plt.figure(figsize=(6, 4))
            plt.plot(years, sales_forecast, marker="o", color="blue")
            plt.title("3-Year Sales Forecast")
            plt.xlabel("Year"); plt.ylabel("Sales INR"); plt.grid(True)
            plt.tight_layout()
            plt.close(fig1)

            # Chart 2: Profit forecast
            fig2 = plt.figure(figsize=(6, 4))
            plt.plot(years, profit_forecast, marker="o", color="purple")
            plt.title("3-Year Profit Forecast")
            plt.xlabel("Year"); plt.ylabel("Profit INR"); plt.grid(True)
            plt.tight_layout()
            plt.close(fig2)

            # Chart 3: Financial Risk Score Distribution
            fig3, ax3 = plt.subplots(figsize=(6, 4))
            ax3.hist(df["Financial_Risk_Score"], bins=10, color="red")
            ax3.set_title("Financial Risk Score Distribution")
            ax3.set_xlabel("Score"); ax3.set_ylabel("Count")
            plt.tight_layout()
            plt.close(fig3)

            # Chart 4: Vendor Score Distribution
            fig4, ax4 = plt.subplots(figsize=(6, 4))
            ax4.hist(df["Vendor_Score"], bins=10, color="green")
            ax4.set_title("Vendor Score Distribution")
            ax4.set_xlabel("Score"); ax4.set_ylabel("Count")
            plt.tight_layout()
            plt.close(fig4)

        # AI Recommendations Summary
        top_skus = df.sort_values(by="Monthly_Sales_INR", ascending=False).head(5)[["SKU_Name","Monthly_Sales_INR","Monthly_Demand_Units"]]

        business_specific_recommendations = ""

        if business_type == "Supermarket":
            business_specific_recommendations = """
**Supermarket Specific Recommendations:**
- Implement advanced inventory management systems to reduce waste and optimize stock levels.
- Analyze customer purchasing patterns to optimize store layout and product placement.
- Strengthen relationships to ensure consistent stock and favorable terms.
"""
        elif business_type == "FMCG":
            business_specific_recommendations = """
**FMCG Specific Recommendations:**
- Optimize distribution channels to ensure wider market reach and timely product delivery.
- Invest in market research to identify opportunities for new product development and market penetration.
- Utilize predictive analytics for more accurate demand forecasting and production planning.
"""
        elif business_type == "Clothing":
            business_specific_recommendations = """
**Clothing Specific Recommendations:**
- Monitor fashion trends closely to adjust inventory and avoid overstocking out-of-season items.
- Enhance online presence and leverage social media for customer engagement and sales.
- Optimize supply chain for faster turnaround times to capture fleeting fashion trends.
"""
        elif business_type == "Electronics":
            business_specific_recommendations = """
**Electronics Specific Recommendations:**
- Keep abreast of technological advancements to manage product lifecycles effectively and plan for new product introductions.
- Focus on customer service and after-sales support to build loyalty in a competitive market.
- Optimize pricing strategies based on competitor analysis and product feature sets.
"""
        else:
            business_specific_recommendations = """
**General Business Recommendations:**
- Conduct regular market analysis to identify new opportunities and competitive threats.
- Focus on customer feedback to improve product/service offerings.
- Explore digital marketing strategies to expand customer reach.
"""

        insights_text = f"""
**DataNetra Insights for {user_name} – {company_name} ({business_type}) in {final_city}, {state}):**

- Average Financial Risk Score: {df["Financial_Risk_Score"].mean():.2f}
- Average Vendor Score: {df["Vendor_Score"].mean():.2f}
- Average Growth Potential Score: {df["Growth_Potential_Score"].mean():.2f}
- Overall MSME Health Score: {msme_health_score:.2f}%

**Top 5 SKUs:**
{top_skus.to_string(index=False)}

**Recommendations:**
- Focus on top SKUs for sales growth.
- Monitor high financial risk MSMEs carefully.
- Ensure reliable vendors are prioritized.
- Optimize inventory and demand planning.

{business_specific_recommendations}
"""

        return (
            f"### 💰 Total Sales\n₹{total_sales:,.2f}",
            f"### 📈 Total Profit\n₹{total_profit:,.2f}",
            f"### 🧠 MSME Health Score\n{msme_health_score:.2f}%",
            f"### 🚀 Growth Potential\n{growth_potential:.2f}",
            fig1, fig2, fig3, fig4, # The 4 plots
            insights_text, # Dashboard insights text
            None # No error message
        )
    except Exception as e:
        # Return default error values for KPIs and charts, and an an error message
        return (
            "### N/A (Error)", "### N/A (Error)", "### N/A (Error)", "### N/A (Error)",
            None, None, None, None,
            "Error generating dashboard insights: Please check your data for numerical issues or unexpected values.", # Insights error for dashboard
            f"Error generating dashboard data: {str(e)}. Please check your data for numerical issues or unexpected values."
        )


# ---------------- MULTI-STEP MSME ONBOARDING ----------------
states = ["Kerala","Tamil Nadu","Karnataka","Telangana","Andhra Pradesh","Delhi"]
states_cities = {
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Other"],
    "Tamil Nadu": ["Chennai", "Madurai", "Coimbatore", "Trichy", "Other"],
    "Karnataka": ["Bangalore", "Mysore", "Mangalore", "Other"],
    "Telangana": ["Hyderabad", "Warangal", "Other"],
    "Andhra Pradesh": ["Visakhapatnam", "Vijayawada", "Guntur", "Other"],
    "Delhi": ["New Delhi", "Other"]
}
business_types = ["FMCG","Supermarket","Clothing","Electronics"]
roles = [
    "Business Owner",
    "Co-Founder",
    "Store Manager",
    "Operations Manager",
    "Finance / Accounts",
    "Analyst"
]
revenue_ranges = [
    "< ₹5 Lakhs",
    "₹5–25 Lakhs",
    "₹25 Lakhs – ₹1 Crore",
    "> ₹1 Crore"
]

# New Global Lists
ownership_types = ["Sole Proprietorship", "Partnership", "Pvt. Ltd. Company", "Public Ltd. Company", "LLP", "Other"]
employee_ranges = ["1-9", "10-49", "50-249", "250+"]

# Using existing kernel state variables for consistency
legal_structures = ["Proprietorship", "Partnership", "Private Limited", "Public Limited", "Limited Liability Partnership (LLP)", "Co-operative Society", "Other"]
agent_teams = ["Sales", "Support", "Marketing", "Operations", "Product Development", "Finance"]
primary_challenges_options = ["Cash Flow Management", "Market Competition", "Talent Acquisition", "Supply Chain Issues", "Technology Adoption", "Regulatory Compliance", "Other"]
expected_support_options = ["Financial Advisory", "Marketing Strategy", "Operational Efficiency", "Technology Solutions", "Funding Assistance", "Export Promotion", "Skill Development"]
key_business_goals = ["Increase Sales", "Improve Profitability", "Expand Market Share", "Cost Reduction", "Product Diversification", "Digital Transformation", "Other"]

# ---------------- APP ----------------
with gr.Blocks(title="MSME Intelligent Agent") as demo:

    step_state = gr.State(0)
    dashboard_insights_state = gr.State("")

    # State variables to hold info across steps
    user_name_state = gr.State("")
    mobile_state = gr.State("")
    company_state = gr.State("")
    business_type_state = gr.State("")
    state_state = gr.State("")
    city_state = gr.State("") # This will store the *actual* city, whether selected or 'Other'
    other_city_input_state = gr.State("") # This will store the raw 'Other' input
    # Assuming these additional fields are not collected in the current UI, they will remain empty states
    legal_structure_state = gr.State("")
    ownership_type_state = gr.State("")
    employee_range_state = gr.State("")
    agent_team_state = gr.State("")
    primary_challenges_state = gr.State("")
    expected_support_state = gr.State("")
    key_business_goals_state = gr.State("")

    # ---------------- LANDING ----------------
    landing = gr.Column(visible=True)
    with landing:
        gr.Markdown("## MSME Intelligent Agent")
        gr.Markdown("""
**AI-powered onboarding and predictive insights for MSMEs**

A secure, AI-driven onboarding and intelligence platform designed to support
**Micro, Small & Medium Enterprises (MSMEs)** through automated classification,
agent mapping, and data-driven business insights.

---
### \U0001f50d What this platform does
- Smart MSME onboarding & identity capture
- Automated business classification & segmentation
- AI-based analysis using uploaded business data
- Professional insight reports for decision-making

---
### \U0001f512 Trust, Security & Compliance
- Fully aligned with **DPDP Act 2023**
- No personal data shared externally
- Built for future integration with Government MSME systems (Udyam, State portals)

---
### \U0001f962 How it works
**Register \u2192 Provide Business Details \u2192 Upload Data \u2192 Get AI Insights**
"""
)
        start_btn = gr.Button("MSME Registered User login", variant="primary")
        gr.Markdown("*For NON MSME Registered user, follow this link https://udyamregistration.gov.in/ to register*")

    # ---------------- STEP 1 ----------------
    step1 = gr.Column(visible=False)
    with step1:
        gr.Markdown("### Step 1 – User Identity")
        full_name = gr.Textbox(label="Full Name *")
        mobile = gr.Textbox(label="Mobile Number *")
        email = gr.Textbox(label="Email")
        role = gr.Dropdown(roles, label="Role *", interactive=True)
        error_step1 = gr.Textbox(label='', interactive=False, value='', elem_classes=['error_message'])
        with gr.Row():
            cancel1 = gr.Button("❌ Cancel", variant="primary")
            next1 = gr.Button("Next \u2192", variant="primary")

    # ---------------- STEP 2 ----------------
    user_status = {"msme_verification": "NOT_STARTED"}

    def step2_submit_logic(msme_checkbox, consent_checkbox, certificate_file):
        if not msme_checkbox:
            return "Please confirm MSME registration ✅", 2 # Stay on current step (step 2)
        if not consent_checkbox:
            return "Please provide consent ✅", 2 # Stay on current step (step 2)
        if certificate_file is None:
            return "Please upload MSME/Udyam certificate 📄", 2 # Stay on current step (step 2)

        # Simulate backend action: Set status to pending approval
        user_status["msme_verification"] = "PENDING_APPROVAL"
        message = "Certificate submitted! 🏛️ Pending verification..."
        time.sleep(2)  # 2 seconds delay
        return message, 3 # Move to step 3

    step2_col = gr.Column(visible=False)
    with step2_col:
        gr.Markdown("## STEP 2 – MSME Declaration & Certificate Upload 🏛️")
        msme_checkbox = gr.Checkbox(label="I declare I am a MSME registered user ☑")
        consent_checkbox = gr.Checkbox(label="I give consent for verification & data usage ☑")
        certificate_file = gr.File(label="Upload MSME / Udyam Certificate 📄")
        submit_btn_step2 = gr.Button("Submit for Verification")
        status_output_step2 = gr.Textbox(value="", interactive=False)

    # ---------------- STEP 3 ----------------
    def step3_verification():
        status = user_status.get("msme_verification", "NOT_STARTED")

        if status == "PENDING_APPROVAL":
            display_message = "🕒 Verification in progress..."
            time.sleep(2)
            user_status["msme_verification"] = "APPROVED"
            status = "APPROVED"

        if status == "APPROVED":
            display_message = "✅ Your MSME verification is approved!"
            time.sleep(2)
            return display_message, gr.update(visible=True)

        return display_message, gr.update(visible=False)

    step3_col = gr.Column(visible=False)
    with step3_col:
        gr.Markdown("## STEP 3 – MSME Verification Status 🏛️")
        status_output_step3 = gr.Textbox(value="Checking status...", interactive=False)
        next_step_btn_step3 = gr.Button("Proceed to enter the business profile details➡️", visible=False)

    # ---------------- STEP 4: Business Profile Details ----------------
    # Predefined states and cities (using the global one for consistency)
    # states_cities = { ... }

    # This function is now redundant as its logic is integrated into update_step4_states_and_proceed
    # def step4_ui_logic(company_name, business_type, state, city, other_city, years_operation, monthly_revenue):
    #     final_city = other_city.strip() if city == "Other" else city or "Not Provided"
    #
    #     result_html = f"""
    # <div style='text-align:center; font-size:20px; color:#333; line-height:1.5;'>
    #     ✅ Business Details Submitted Successfully!<br><br>
    #     <b>Company Name:</b> {company_name}<br>
    #     <b>Business Type:</b> {business_type}<br>
    #     <b>State:</b> {state}<br>
    #     <b>City:</b> {final_city}<br>
    #     <b>Years of Operation:</b> {years_operation}<br>
    #     <b>Monthly Revenue Range:</b> {monthly_revenue}<br>
    # </div>
    # """
    #     return result_html

    step4_col = gr.Column(visible=False)
    with step4_col:
        gr.Markdown("## STEP 4 – Business Profile Details 🏢")
        company_name_step4 = gr.Textbox(label="Company Name", placeholder="Enter your company name")
        business_type_step4 = gr.Dropdown(label="Business Type", choices=["FMCG", "Supermarket", "Clothing", "Electronics"])
        state_step4 = gr.Dropdown(label="State", choices=list(states_cities.keys()))
        city_step4 = gr.Dropdown(label="City", choices=["Select state first"])
        other_city_step4 = gr.Textbox(label="If Other, specify city", placeholder="Enter city", visible=False)
        years_operation_step4 = gr.Textbox(label="Years of Operation", placeholder="e.g., 5")
        monthly_revenue_step4 = gr.Dropdown(label="Monthly Revenue Range", choices=["<1 Lakh", "1-5 Lakh", "5-10 Lakh", ">10 Lakh"])
        submit_btn_step4 = gr.Button("Submit Business Profile ✅")
        output_html_step4 = gr.HTML("")

    # ---------------- STEP 5 ----------------
    step5_consent_upload = gr.Column(visible=False)
    with step5_consent_upload:
        gr.Markdown("### Step 5 – Consent & File Upload")
        consent = gr.Checkbox(label="✅ I consent to use my data for AI analysis (mandatory) *")
        file_upload = gr.File(label="Upload Retail Excel File *")
        upload_status = gr.Textbox(label="Upload Status", interactive=False)
        error_step5 = gr.Textbox(label='', interactive=False, value='', elem_classes=['error_message'])
        with gr.Row():
            back5 = gr.Button("\u2190 Back", variant="primary")
            cancel5 = gr.Button("❌ Cancel", variant="primary")
            submit_btn_step5 = gr.Button("🚀 Submit & Analyze", variant="primary") # Renamed
        insights_output = gr.Markdown()
        pdf_output = gr.File(label="Download PDF Report")
        view_dashboard_btn = gr.Button("📊 View Dashboard", variant="primary", visible=False)

    # ---------------- STEP 6 : DASHBOARD (Enhanced) ----------------
    step6_dashboard = gr.Column(visible=False)
    with step6_dashboard:
         gr.Markdown("## \U0001f4ca Business Performance Dashboard")

         with gr.Row():
              kpi_sales = gr.Markdown("### \U0001f4b0 Total Sales\n—")
              kpi_profit = gr.Markdown("### \U0001f4c8 Total Profit\n—")
              kpi_health = gr.Markdown("### \U0001f9e0 MSME Health Score\n—")
              kpi_growth = gr.Markdown("### \U0001f680 Growth Potential\n—")

         gr.Markdown("---")

         with gr.Row():
             chart1 = gr.Plot(label="Chart 1")
             chart2 = gr.Plot(label="Chart 2")

         with gr.Row():
             chart3 = gr.Plot(label="Chart 3")
             chart4 = gr.Plot(label="Chart 4")

         gr.Markdown("---")
         dashboard_insights_markdown = gr.Markdown("### AI Recommendations")

         back_to_step5_btn = gr.Button("\u2b05 Back to AI Insights & PDF", variant="primary")


    # Function to update city dropdown based on state selection
    def update_cities(state):
        if state in states_cities:
            return gr.update(choices=states_cities[state], value=None), gr.update(visible=False)
        else:
            return gr.update(choices=[], value=None), gr.update(visible=False)

    def show_other_city_textbox(city):
        if city == "Other":
            return gr.update(visible=True)
        else:
            return gr.update(visible=False)

    # ---------------- NAVIGATION ----------------
    def update_visibility(current_step):
        return (
            gr.update(visible=current_step==0), # landing
            gr.update(visible=current_step==1), # step1
            gr.update(visible=current_step==2), # step2_col
            gr.update(visible=current_step==3), # step3_col
            gr.update(visible=current_step==4), # step4_col
            gr.update(visible=current_step==5), # step5_consent_upload
            gr.update(visible=current_step==6)  # step6_dashboard
        )

    def next_step_fn(step):
        return min(step+1, 6)

    def prev_step_fn(step):
        return max(step-1, 0)

    def reset_fn(_):
        return 0

    # Validation function for Step 1
    def validate_step1(current_step, full_name_val, mobile_number_val, role_val):
        if not full_name_val or not mobile_number_val or not role_val:
            return "Please fill in Full Name, Mobile Number, and Role.", current_step
        return "", current_step + 1

    # No need for validate_step2, validate_step3, validate_step4_details explicitly now, as logic is in submit handlers

    # Validation and Submission function for Step 5 (formerly Step 4)
    def handle_submission(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, consent_checked, uploaded_file):
        if not consent_checked:
            return "Please check the consent box to proceed.", None, None, "### N/A", "### N/A", "### N/A", "### N/A", None, None, None, None, gr.update(visible=False), ""
        if uploaded_file is None:
            return "Please upload an Excel file.", None, None, "### N/A", "### N/A", "### N/A", "### N/A", None, None, None, None, gr.update(visible=False), ""

        df = None
        try:
            df = pd.read_excel(uploaded_file.name)
        except Exception as e:
            return f"Error reading Excel file: {str(e)}. Please ensure it's a valid Excel format.", None, None, "### N/A", "### N/A", "### N/A", "### N/A", None, None, None, None, gr.update(visible=False), ""

        # Centralized column validation
        required_columns = get_required_columns(business_type)
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            error_message = f"Missing required columns in the uploaded file for '{business_type}' analysis: {', '.join(missing_columns)}. Please ensure your file contains all necessary data for analysis."
            return error_message, None, None, "### N/A", "### N/A", "### N/A", "### N/A", None, None, None, None, gr.update(visible=False), ""

        # Generate insights
        insights_result, insights_error = generate_insights(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df.copy())

        # Generate PDF
        pdf_path_result, pdf_error = generate_pdf(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df.copy())

        # Generate dashboard data
        kpi_sales_text, kpi_profit_text, kpi_health_text, kpi_growth_text, fig1, fig2, fig3, fig4, dashboard_insights_text, dashboard_error = \
            generate_dashboard_data(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df.copy())

        # Consolidate errors
        final_error_message = ""
        if insights_error:
            final_error_message += f"AI Insights: {insights_error}\n"
        if pdf_error:
            final_error_message += f"PDF Report: {pdf_error}\n"
        if dashboard_error:
            final_error_message += f"Dashboard Data: {dashboard_error}\n"

        if final_error_message:
            # If any error, return consolidated error message and hide dashboard button
            return final_error_message, None, None, "### N/A", "### N/A", "### N/A", "### N/A", None, None, None, None, gr.update(visible=False), ""

        # If all successful, return results and make dashboard button visible
        return "", insights_result, pdf_path_result, kpi_sales_text, kpi_profit_text, kpi_health_text, kpi_growth_text, fig1, fig2, fig3, fig4, gr.update(visible=True), dashboard_insights_text

    # New function to update and show the dashboard
    def update_and_show_dashboard(
        user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, file, stored_dashboard_insights_value
    ):
        # Read file again for dashboard, this could be optimized by passing df_copy from handle_submission
        # But for now, keeping it separate for clarity in this function's scope
        df = None
        if file:
            try:
                df = pd.read_excel(file.name)
            except Exception:
                pass # Error handled in handle_submission, this is for chart generation

        if df is None:
             return (
                "### N/A (Error)", "### N/A (Error)", "### N/A (Error)", "### N/A (Error)",
                None, None, None, None,
                "Error loading dashboard insights.", # Return error for markdown
                5 # Stay on step 5 if there's an error getting data
            )

        kpi_sales_text, kpi_profit_text, kpi_health_text, kpi_growth_text, fig1, fig2, fig3, fig4, dashboard_insights_text, dashboard_error = \
            generate_dashboard_data(user_name, mobile_number, company_name, state, city, other_city_name, business_type, legal_structure, ownership_type, employee_range, agent_team, primary_challenges, expected_support, key_business_goals, df.copy())

        # If there's an error from generate_dashboard_data at this point, it means some internal calculation failed.
        # We should display error KPIs and no charts.
        if dashboard_error:
            return (
                kpi_sales_text, kpi_profit_text, kpi_health_text, kpi_growth_text,
                None, None, None, None, # Charts should be None on error
                dashboard_insights_text, # Return the error message for the insights markdown
                5 # Stay on step 5
            )

        # Return updates for visibility and dashboard components
        return (
            kpi_sales_text,
            kpi_profit_text,
            kpi_health_text,
            kpi_growth_text,
            fig1, fig2, fig3, fig4,
            stored_dashboard_insights_value, # Use the stored value for dashboard insights
            6 # Move to step 6
        )

    # Connect events
    start_btn.click(next_step_fn, step_state, step_state).then(
        update_visibility, step_state, [landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Update user_name_state and mobile_state when next1 is clicked
    next1.click(
        fn=lambda name, mob: (name, mob), # Update state vars
        inputs=[full_name, mobile],
        outputs=[user_name_state, mobile_state],
        queue=False # Run this quickly
    ).then(validate_step1,
                inputs=[step_state, full_name, mobile, role],
                outputs=[error_step1, step_state]) \
         .then(update_visibility, step_state,
               outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard])

    cancel1.click(reset_fn, step_state, step_state).then(
        update_visibility, step_state, [landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 2 submit
    submit_btn_step2.click(
        fn=step2_submit_logic,
        inputs=[msme_checkbox, consent_checkbox, certificate_file],
        outputs=[status_output_step2, step_state] # status_output_step2 and step_state are outputs
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 3 activation (triggered by step_state change to 3)
    step_state.change(
        fn=lambda current_step_value: step3_verification() if current_step_value == 3 else (gr.update(value=""), gr.update(visible=False)),
        inputs=step_state,
        outputs=[status_output_step3, next_step_btn_step3]
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    ) # Added .then() to update visibility after step3_verification

    next_step_btn_step3.click(
        fn=lambda: 4, # Go to step 4
        outputs=step_state
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 4 logic
    def update_city_options_step4_internal(selected_state):
        cities = states_cities.get(selected_state, [])
        visible_other = "Other" in cities
        return gr.update(choices=cities, value=cities[0] if cities else ""), gr.update(visible=visible_other)

    state_step4.change(fn=update_city_options_step4_internal, inputs=state_step4, outputs=[city_step4, other_city_step4])

    # Function to update all relevant state variables from Step 4 inputs and proceed
    def update_step4_states_and_proceed(comp_name, b_type, st, cty, other_cty, yrs_op, monthly_rev):
        final_city = other_cty if cty == "Other" else cty
        success_message = f"""
    <div style='text-align:center; font-size:20px; color:green; line-height:1.5;'>
        ✅ Registered successfully!<br><br>
        <b>Company Name:</b> {comp_name}<br>
        <b>Business Type:</b> {b_type}<br>
        <b>State:</b> {st}<br>
        <b>City:</b> {final_city}<br>
        <b>Years of Operation:</b> {yrs_op}<br>
        <b>Monthly Revenue Range:</b> {monthly_rev}<br>
    </div>
    """
        # Return all states to be updated and the message
        return comp_name, b_type, st, final_city, other_cty, success_message

    submit_btn_step4.click(
        fn=update_step4_states_and_proceed,
        inputs=[company_name_step4, business_type_step4, state_step4, city_step4, other_city_step4, years_operation_step4, monthly_revenue_step4],
        outputs=[company_state, business_type_state, state_state, city_state, other_city_input_state, output_html_step4] # Add output_html_step4 here
    ).then(
        # Now update step_state after a short delay
        fn=lambda current_step: (current_step + 1),
        inputs=step_state,
        outputs=step_state
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 5 navigation
    back5.click(prev_step_fn, step_state, step_state).then(
        update_visibility, step_state, [landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )
    cancel5.click(reset_fn, step_state, step_state).then(
        update_visibility, step_state, [landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # ---------------- FILE UPLOAD ----------------
    def file_upload_status(file, name):
        if file is not None and name:
            return f"Thank you {name} for uploading your file! ✅ Click Submit & Analyze to view AI Insights, Dashboard and generate PDF Report."
        return ""

    file_upload.upload(file_upload_status, inputs=[file_upload, full_name], outputs=upload_status)

    # ---------------- ANALYSIS ----------------
    submit_btn_step5.click(
        handle_submission,
        inputs=[
            user_name_state, mobile_state, company_state, state_state, city_state, other_city_input_state, business_type_state,
            legal_structure_state, ownership_type_state, employee_range_state, agent_team_state, primary_challenges_state, expected_support_state, key_business_goals_state,
            consent, file_upload
        ],
        outputs=[
            error_step5, insights_output, pdf_output,
            kpi_sales, kpi_profit, kpi_health, kpi_growth,
            chart1, chart2, chart3, chart4, view_dashboard_btn, dashboard_insights_state
        ]
    ).then(
        lambda: 5, # Stay on step 5 (to see insights/pdf)
        outputs=step_state
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 5 (Consent & Upload) \u2192 Step 6 (Dashboard)
    view_dashboard_btn.click(
        update_and_show_dashboard, # Call the new function to update and show dashboard
        inputs=[
            user_name_state, mobile_state, company_state, state_state, city_state, other_city_input_state, business_type_state,
            legal_structure_state, ownership_type_state, employee_range_state, agent_team_state, primary_challenges_state, expected_support_state, key_business_goals_state,
            file_upload, dashboard_insights_state
        ],
        outputs=[
            kpi_sales, kpi_profit, kpi_health, kpi_growth,
            chart1, chart2, chart3, chart4, # Outputs for the dashboard components
            dashboard_insights_markdown, # Added dashboard_insights_markdown as output
            step_state # Update step_state to 6
        ]
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Step 6 (Dashboard) \u2192 Step 5 (Consent & Upload)
    back_to_step5_btn.click(
        fn=lambda: 5, # Set step_state to 5
        outputs=step_state
    ).then(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    # Centralized step_state change handler for overall visibility
    # This will be triggered on initial load as step_state is initialized to 0
    step_state.change(
        fn=update_visibility,
        inputs=step_state,
        outputs=[landing, step1, step2_col, step3_col, step4_col, step5_consent_upload, step6_dashboard]
    )

    demo.load(
        fn=lambda: 0, # Initialize step_state to 0, which will trigger the .change event above
        inputs=None,
        outputs=[step_state]
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ff2d109c870986a43f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
